# Faster R-CNNで実装

In [1]:
# 必要なライブラリのインストール
!pip install -q lightning torchmetrics pycocotools

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 20.1 MB/s eta 0:00:0000:01


In [2]:
import os
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import v2 as transforms
from torchvision.utils import draw_bounding_boxes

from pycocotools.coco import COCO
from torchmetrics.detection import MeanAveragePrecision
import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint


# Datasetクラスの定義
- COCO形式のアノテーションからFaster R-CNN用ターゲット形式（`[x1, y1, x2, y2]`）への変換
- `transforms.v2` を用いた前処理（非推奨の `ToTensor` ではなく `ToImage` + `ToDtype` を採用）
- 境界値チェック（幅・高さが正、画像範囲内のクリッピング）とアノテーションなし画像の安全な処理
- 可変長バッチに対応する `collate_fn`


In [3]:
class PlantDataset(Dataset):
    def __init__(self, data_dir, coco_annotation, transform=None):
        super().__init__()
        self.image_root = data_dir
        self.annotation_path = coco_annotation
        self.coco = COCO(self.annotation_path)
        self.ids = list(sorted(self.coco.imgs.keys()))
        
        # torchvision v2 推奨の前処理
        if transform is not None:
            self.transform = transform
        else:
            self.transform = transforms.Compose([
                transforms.ToImage(),
                transforms.ToDtype(torch.float32, scale=True)
            ])

    def __getitem__(self, index):
        image_id = self.ids[index]
        img_info = self.coco.loadImgs(image_id)[0]

        image_path = os.path.join(self.image_root, img_info["file_name"])
        img = Image.open(image_path).convert("RGB")

        ann_ids = self.coco.getAnnIds(imgIds=image_id)
        anns = self.coco.loadAnns(ann_ids)

        target = self._build_target(image_id, anns, img_info["width"], img_info["height"])

        if self.transform:
            img = self.transform(img)

        return img, target

    def _build_target(self, img_id, anns, width, height):
        boxes = []
        labels = []
        areas = []
        iscrowd = []

        for ann in anns:
            x, y, w, h = ann["bbox"]
            if w <= 0 or h <= 0:
                continue
            x1 = max(0.0, float(x))
            y1 = max(0.0, float(y))
            x2 = min(float(width), float(x + w))
            y2 = min(float(height), float(y + h))

            if x2 > x1 and y2 > y1:
                boxes.append([x1, y1, x2, y2])
                labels.append(ann["category_id"])
                areas.append(ann.get("area", (x2 - x1) * (y2 - y1)))
                iscrowd.append(ann.get("iscrowd", 0))

        if len(boxes) > 0:
            target = {
                "boxes": torch.as_tensor(boxes, dtype=torch.float32),
                "labels": torch.as_tensor(labels, dtype=torch.int64),
                "image_id": torch.tensor([img_id]),
                "area": torch.as_tensor(areas, dtype=torch.float32),
                "iscrowd": torch.as_tensor(iscrowd, dtype=torch.int64)
            }
        else:
            target = {
                "boxes": torch.zeros((0, 4), dtype=torch.float32),
                "labels": torch.zeros((0,), dtype=torch.int64),
                "image_id": torch.tensor([img_id]),
                "area": torch.zeros((0,), dtype=torch.float32),
                "iscrowd": torch.zeros((0,), dtype=torch.int64)
            }

        return target

    def __len__(self):
        return len(self.ids)

    @staticmethod
    def collate_fn(batch):
        return tuple(zip(*batch))


# LightningModuleでモデル・学習・検証・テストを定義
- Faster R-CNN (ResNet50 FPN) の転移学習
- 検証時に `val_loss` の計算に加え、`MeanAveragePrecision` を正しく更新（`update`）して算出
- 検出モデルの学習に最適な SGD + MultiStepLR スケジューラを設定


In [5]:
class PlantModel(L.LightningModule):
    def __init__(self, num_classes=2, lr=0.005, momentum=0.9, weight_decay=0.0005):
        super().__init__()
        self.save_hyperparameters()
        
        # 事前学習済み Faster R-CNN (ResNet50-FPN) の取得
        self.model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")
        
        # クラス数 (背景1 + 物体1 = 2) に合わせてヘッドを置換
        in_features = self.model.roi_heads.box_predictor.cls_score.in_features
        self.model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

        # 評価指標 (COCO mAP)
        self.map_metric = MeanAveragePrecision(box_format="xyxy")

    def forward(self, images, targets=None):
        return self.model(images, targets)

    def training_step(self, batch, batch_idx):
        imgs, targets = batch
        loss_dict = self.model(imgs, targets)
        loss = sum(loss_dict.values())
        
        self.log("train_loss", loss, on_step=True, on_epoch=True, prog_bar=True, logger=True, batch_size=len(imgs))
        for k, v in loss_dict.items():
            self.log(f"train_{k}", v, on_epoch=True, batch_size=len(imgs))
        return loss

    def validation_step(self, batch, batch_idx):
        imgs, targets = batch
        
        # 1. 損失の計算 (Faster R-CNNはtrainモードでloss_dictを返すため一時的にtrainモードへ)
        self.model.train()
        with torch.no_grad():
            loss_dict = self.model(imgs, targets)
            val_loss = sum(loss_dict.values())
        self.log("val_loss", val_loss, on_step=False, on_epoch=True, prog_bar=True, logger=True, batch_size=len(imgs))

        # 2. 予測の取得とmAPの更新 (evalモードで推論を実行しメトリクスを更新)
        self.model.eval()
        with torch.no_grad():
            preds = self.model(imgs)
            self.map_metric.update(preds, list(targets))

        return val_loss

    def on_validation_epoch_end(self):
        metrics = self.map_metric.compute()
        self.log("val_mAP", metrics["map"], prog_bar=True, logger=True)
        self.log("val_mAP_50", metrics["map_50"], prog_bar=True, logger=True)
        self.log("val_mAP_75", metrics["map_75"], logger=True)
        self.map_metric.reset()

    def test_step(self, batch, batch_idx):
        imgs, targets = batch
        self.model.eval()
        with torch.no_grad():
            preds = self.model(imgs)
            self.map_metric.update(preds, list(targets))
        return preds

    def on_test_epoch_end(self):
        metrics = self.map_metric.compute()
        self.log("test_mAP", metrics["map"], logger=True)
        self.log("test_mAP_50", metrics["map_50"], logger=True)
        self.log("test_mAP_75", metrics["map_75"], logger=True)
        self.map_metric.reset()

    def configure_optimizers(self):
        # Faster R-CNNで安定して収束する SGD + モーメンタム
        params = [p for p in self.parameters() if p.requires_grad]
        optimizer = torch.optim.SGD(
            params,
            lr=self.hparams.lr,
            momentum=self.hparams.momentum,
            weight_decay=self.hparams.weight_decay
        )
        # エポック16と24で学習率を0.1倍にするスケジューラ (30エポック向け)
        lr_scheduler = torch.optim.lr_scheduler.MultiStepLR(
            optimizer,
            milestones=[16, 24],
            gamma=0.1
        )
        return [optimizer], [lr_scheduler]


# データセットとDataLoaderの設定


In [9]:
train_dir = "/kaggle/input/datasets/data4learning/plant-data/train"
train_ann = "/kaggle/input/datasets/data4learning/plant-data/train/_annotations.coco.json"

val_dir = "/kaggle/input/datasets/data4learning/plant-data/valid"
val_ann = "/kaggle/input/datasets/data4learning/plant-data/valid/_annotations.coco.json"

test_dir = "/kaggle/input/datasets/data4learning/plant-data/test"
test_ann = "/kaggle/input/datasets/data4learning/plant-data/test/_annotations.coco.json"

train_dataset = PlantDataset(data_dir=train_dir, coco_annotation=train_ann)
val_dataset = PlantDataset(data_dir=val_dir, coco_annotation=val_ann)
test_dataset = PlantDataset(data_dir=test_dir, coco_annotation=test_ann)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=4,
    collate_fn=PlantDataset.collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=4,
    collate_fn=PlantDataset.collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=4,
    collate_fn=PlantDataset.collate_fn
)

print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}, Test samples: {len(test_dataset)}")


loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Train samples: 656, Val samples: 164, Test samples: 54


# モデルの学習（4エポック）
- `ModelCheckpoint` コールバックにより、検証時 `val_mAP` が最高スコアの重みを自動保存
- `max_epochs=30` で学習を実行


In [11]:
# 最良モデルを保存するコールバック (val_mAP を監視)
checkpoint_callback = ModelCheckpoint(
    dirpath="/kaggle/working/checkpoints",
    filename="plant-fasterrcnn-{epoch:02d}-{val_mAP:.3f}",
    monitor="val_mAP",
    mode="max",
    save_top_k=1,
    verbose=True
)

model = PlantModel(num_classes=2, lr=0.005)

# エポック数 30 で Trainer を設定
trainer = L.Trainer(
    max_epochs=4,
    accelerator="auto",
    devices=1,
    callbacks=[checkpoint_callback],
    log_every_n_steps=10
)

# 学習の実行
trainer.fit(model, train_loader, val_loader)

print(f"Best checkpoint saved at: {checkpoint_callback.best_model_path}")


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type                 ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model      │ FasterRCNN           │ 41.3 M │ train │     0 │
│ 1 │ map_metric │ MeanAveragePrecision │      0 │ train │     0 │
└───┴────────────┴──────────────────────┴────────┴───────┴───────┘

Trainable params: 41.1 M                                                                                           
Non-trainable params: 222 K                                                                                        
Total params: 41.3 M                                                                                               
Total estimated model params size (MB): 165.197                                                                    
Modules in train mode: 190                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

Epoch 0, global step 164: 'val_mAP' reached 0.42826 (best 0.42826), saving model to '/kaggle/working/checkpoints/plant-fasterrcnn-epoch=00-val_mAP=0.428.ckpt' as top 1
Epoch 1, global step 328: 'val_mAP' reached 0.48472 (best 0.48472), saving model to '/kaggle/working/checkpoints/plant-fasterrcnn-epoch=01-val_mAP=0.485.ckpt' as top 1
Epoch 2, global step 492: 'val_mAP' reached 0.54989 (best 0.54989), saving model to '/kaggle/working/checkpoints/plant-fasterrcnn-epoch=02-val_mAP=0.550.ckpt' as top 1
Epoch 3, global step 656: 'val_mAP' reached 0.55797 (best 0.55797), saving model to '/kaggle/working/checkpoints/plant-fasterrcnn-epoch=03-val_mAP=0.558.ckpt' as top 1
`Trainer.fit` stopped: `max_epochs=4` reached.


Best checkpoint saved at: /kaggle/working/checkpoints/plant-fasterrcnn-epoch=03-val_mAP=0.558.ckpt


# テストデータセットでの評価
学習で得られた最良重み（Best Checkpoint）を自動ロードしてテストデータセット全体の mAP を測定します。


In [12]:
# 最良チェックポイントによるテスト評価
trainer.test(model, test_loader, ckpt_path="best")


Restoring states from the checkpoint path at /kaggle/working/checkpoints/plant-fasterrcnn-epoch=03-val_mAP=0.558.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at /kaggle/working/checkpoints/plant-fasterrcnn-epoch=03-val_mAP=0.558.ckpt


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_mAP          │     0.567171573638916     │
│        test_mAP_50        │    0.7873457074165344     │
│        test_mAP_75        │    0.6898621320724487     │
└───────────────────────────┴───────────────────────────┘

[{'test_mAP': 0.567171573638916,
  'test_mAP_50': 0.7873457074165344,
  'test_mAP_75': 0.6898621320724487}]